# QosTil AI — Kazakh, Russian & English language detection

**Veritas AI International High School Hackathon 2026 · AI + Languages**

An *actual trained machine-learning model*, not a chatbot wrapper, predicts the language of each word in mixed-language school notices. Demonstrates the model, a simple rules-only baseline, per-language metrics, a harder synthetic challenge set, and known limitations.

**Data provenance:** The word seed list and both evaluation sets are *AI-assisted, synthetic examples written for this prototype*. They are **not** representative of real messages. No real student personal data was collected. This is an educational prototype, not an independently validated production service. All numbers below are calculated by running these cells (never assumed).

## 1. Load the small bundled synthetic dataset
All dataset bytes are stored **inside this notebook**, so reviewers do not need any external download or permission. The compressed strings below decode into three readable text files. For transparency, the original text is available by evaluating `seed_text`, `heldout_text` and `challenge_text` in Python.

Use **Runtime → Run all** in Google Colab. The needed libraries are normally preinstalled; if not, run `%pip install scikit-learn numpy`.

In [1]:
import zlib, base64, re

def unpack(encoded):
    return zlib.decompress(base64.b85decode(encoded)).decode('utf-8')

seed_text = unpack(
    'c-nPZ%aS5VcHP%kxY;bU(ewvMtJ%m*3rQROKmbk>lR*R%kD`(FB1u$TIjZVPYa6pN8>?lnW;H-ULh;I9fb1XAIp=yn(A{HI84-'
    'TN{rY{J`*7fQ?hmQU^7tlBqvW66AO8LKQBlN2={v5KWo1$3(ZCI}yyMzglK6HRXQ^xFKH`jSmb<bS7jEeNtxr4uRJcK&m039&_;2'
    '3)&iz}K_oMP(ToR=>RdnOO8}{PT7lWwne_!}Pl#2_w7vH%5h(_2WEu%PfaoP=gMTvVG<#Cjj#lO1${ICCZk<0xc_l3vIGJfl^CLd'
    'JAH*aolzdO!e|9*TreLn8pae2Bs-5-~)Ke^)ugNNhNVLW~P>DO;IdWWU^;{uPbUyjq`>h$?^rwPZe#&GsJ!@>^@-'
    'eEmnbI%udnt=b8(}&Z&JB`6(#-Tgzz+=Ptm~z~`&R+k5cU!F3x#I!zCt&dbD^B-'
    'd00EABG2mAC!<rfA+aG0`QV;ywAE)lP`1!BL7c!T$5Ss5^Ob4#n=}}v*%m80_b<p;!v&r1*bO$lPZh72+xIVoje+WUHO^avjI}z{'
    'W?&rTkRvAU|>m@8U!GfRvcKnKsmtczF{pkY?zTnbeRtfMx0nS5kp<%ItbaCF&4%m5d`Wy%}7yecx(Q{MTY8vFG_ptsmJxIrCZ<&2'
    '8XF<dlyxo{nA)GvSM~ep?u)_z6vLOt#<qCF(ix;#M*?{+q;*14}CIRNSYY@2ZYR2*Tb!K8VX|;G4!kR9nAg8et5O$a0PWD&|tC)J'
    'g$`^yfOzt&X$m8@QW(YPrAV_{c2qjZO_8`8*I=nDDK4S`yIHPTjPh>xX6Sv$;C=pE$!sZ29Tu(n>{niHDbxW*;M^PCHqyf~}7&>C'
    'beURlGbHd^MnW;7-mkV!3qT%W@Cup0~1H+9w9A1BV{R<vV4FZ92>15-20Ej$X-aHG5W3Jf*+tS7o5?O+F1C~Az05gLpHD6l-'
    'z;+Z8<|7XXGIq6plxf8MCj}I-rp+P!Mg|lNCBp`WesKy-DuSJ7W+(b`r4U4T3khVwDJ#L0fyL}_dT7{4Uc}NzqW=R8B13Ma1Ro{4'
    '-1i(bg;YDUkh|ngUK{z-'
    'h9QUAU{;5Ra)cmk@wzhw50<KEKg6m6^BlMb%FPGO70p=KYhz8wg=IoyU}(_Vg?zR@L+clN)?Ao9VBuArJHh}e1vvwz^mcy*u}~b&'
    'Z&zm8yAV?Q^N<LV+>N&@<rF}DSEGdk33j2cLe{)+w4tV1KK)=YR6JNJ)sP-kRt|50B0?U+&zB+d-'
    'u;F6jN42;ZO*%tCeztoOj(ZtbH(KBBFWO=VP>qSKUz&`h(~!XOVq>}wFL2{AvXe&gk|7V&ylKM%!ylxxKOcdB>CBI5r2!Im|WDD-'
    'G%{n+J`bKbIWA)#Cj0iUh7gjR^^RxReLRn&GHRad)7v{_}L)cD3(^t&IJ;Roe8UExeJ75<=Y3)b4P=mGJ_IbDE?=#GhnQ;nL6hV<'
    '0^!jU5JI)f;EMj)^i$dMq{a};hmJ5GzMpTHAuBTu6vBo#r%rASKNkp-Zo2{4b?zcqoCwxo<k0|I;gy<wq<r*2`w!@tFBo-'
    'w9QmhFZoZ>ZW?V)=nx))3pVF0WsVC(#o|P6IA=M|eF%-AwO@=N%hBqQ>Z(pf>Rx4f?vo&<<+6vmeVm3ObMGv11J+|<V-'
    'sL)hW>0gqKr`PbdS>UtO!_#PNptV)-'
    '@7?Tc_S<8fuBm_%75z_B94niF}r<nTHN6K$a&Ljm%=oH<OL&w6n@?o<3L;(Ja6+^4jtV>pDQJq^c4|1hN8hP?vIRldM6vLSPNY!S'
    'G>uiLvrGMe(K%N^5px-hC+iW0h^FyywHx!pIq?n(5XlNGOZ~5hmG%n4DbtqOTzZT7&h)JQ?afYqU1l+ZHx;ug*6}wGXEiv|);H1$'
    's!TW5K3wb1-q@>;z++jVcVh$*b>#*eA%nzz&a2wPp^Z-f^?}>AkED)h41H$YBne(GMo-'
    ';q?CW@${)~d*55H^Mcna1(p1>#}tV?2Ynv0=VLu>K{igVJXmaO%=x6fhrijWZ%-'
    'd>$jq@860XieShH2?8$0ozq0M0rDW;2>$#Jc|-*f~5K5`xTJ!z^o*qEs&LgSOpP->?);gNL6NCz#=W(&D{O>KX`G?i>?Am-'
    '|;@3@}OxR?Fz&u0%lzSqE(!ys!=KVtd8$uo1NG)hnP$FRbLP_QAz9ZRq^&uUl0(<A!>hVn~@9g4CbUT3HOgKfrApx#zinro)*1qm'
    'AOgGTgjt*E(T7MwxjC>QMQpg|5VijAc_1n<=|=R9l(#=(m!tTaDybswa7qEPf8rk%c5e8D3#rUg41%*`hPxh39@>ItRb-'
    'C0fvrmgG<XSz-U>egnUt%aXi;mO)GD-'
    'K3d5A@5E5hsG@3(E_|QI4=vKSjJ~eW+1`*5C{vc{gCQk$UGA{EP7J$;>jf170{984RVdYZ&;^1rZ)k-'
    'y3!oLX6s$$?4PS&w@$7!yMrFdU3`<hkY6I-'
    'q7^U=s=rRQ6)qVgxQCcIwgdG*>sMnQdvapeYnioo_>T>n_7qichH$u+~qgnCrsZfQy3%bI*e{5hUrA++|{_lzJd2nYoBO$dwXaaG'
    '&q3;=*QX$8*9|DJP>GU>zW9zppSj*p)F>iItQ85f(H19$rCwxR%`Pl1J)YJYsBe0%S|d0UoBg#1e9k>iC6hqIY-'
    '+!^7Msx&es>xj(DHzSV58t7*voQf)iPw_jX&F(UzwlL-'
    '<>snpruTTKd;*&5NDS9>XN!Prau}t?`S}Nl@|^LyHuVh|kO8kG6r))J_MQ<x#5#ETZ)zi%(5mY9a;?c7P3!R;s39xV{=l5KHfbW8'
    '*Q1D0J@VK+BGScs+~m4N(Aap6Bd@xyGE01)CA}p$iD_=gfVVA-'
    '%lOark|6+d5<$vu5h$8`~}fp$4bD%%s*~qBQZT&d+O=Gka&GkaiEQ9;&Umlnq=z(YD*uC#$kpTgN|aG$te(2BYm4R!!-'
    'tJ9Xd(eX^#Mw^CyMC{Ny-mo1tOc1E&tW65OB+-KUnzf>oxzY)%x4T6w*)j)irAk5h-'
    'Z1>2wSG|Fe!UD_~*O2~4BqhW#vv6nvNNf!B*&&J-'
    '1={e<WdJqSN(fJuZ2ng+rHQ_0e`|G&HvjW=kq$}L`pvnD{KsGA4W*`!jebKJbZ+VC@_H{hm)C2UgI8=?ln^g3lnNiNg;}5BW8F!R'
    'uhBO!-O<^1E?{&bh2YQcFD>0zCFz|zSMHU0x@NLgSED+1{erC(KT8k&7gMd(PC0+21FCij`YWNngiJppz<C_AnAHr3__djr=5-'
    'P|n9E*dVIDUo>a|sh*WfZYThvi)m49U=xlZOR-'
    'u2_A{)o*GrOf6+{$BDpw6KD1xn~>HvX5!Es&p2eryoSxg<M$=)Keau6%NZH&fx0!e<_}8WA|je7jp`l(I4uEdkY^Pu<lLXjnmwv^'
    '+8YJLsxB7@Asnoyp%jjqYY{I;ZyYnX>MmKGsdE>jO&`I;mN)=2&uo$N~Wl@7FvE@n^=1_sb+40aj8!8U~ri+U@ow*dFNU#-'
    'pxk%Ty%`UymvigqX@SKQtg34DD-'
    'B``%+GylTPi^h5fHO@_JRHfYA4Lq6+PP=%#oCH_M)r>!n0at^oEAbiFR2+jEE8AT<+M{dK@Tod1l^G}L_AejUs){jH<KCN%2JBi|'
    '}~4yxnR8=rnxW}Rr{%B-K|c{X$-Uz9#434}CGZ=8SYbx-'
    '4)Ug=WINg_;fKFX6g+()9hGVXhq_@c;CSG0RsmblVK?H*<oW!3R%>DozD6nU2QUAq@2o!qBx5aC)DU|srFmfbpC26{8V(pDDb9oN'
    'hHewgLAj^BUYq<u2;-'
    '$s4+rwStTThAyhhL~RZZ%dc^s6(NLd0g^d419FUhlJmqnqKJ*2EFr_pRUZo0eX;^|8|h@i%!VizIE9kPD!PN;x7r?QR351l*3Y~Y'
    'zRSkANo#ePi3#l_0Y{@-ioNKWZR<BioW-fu9qcxk3Ca<Lwl1$FWmp=+a)d7cQ<)f4dA64U&@-'
    '{%UKwnc5&^v9J#k~l2DI4i)gM2g2ibCiA^YfOF_#XTnw30TB#f7MG0nYD8Mg@Vdk4G!-d};ySP8d@-'
    'm{+fDMY$iM!pnT_v!5R6vWkA1B~dr7@fUmP3~%9oLFs?|zgdt{<h)9b}l(r-i-'
    '^?)hloT0W}ExLYMIPJtyl5N7yO6?+I3QKkeqgIL5KJxza(utOW*igH*g0}5IRb&C_zE*Y7AxJPMPVSm!yD662F$IB6)bH_gr;@t6'
    '@aat1wtP0Zlm{j<rtBHDO%euG}yi&-U4FCB+k%TXA=nBws39(UlP<zm!07S4!G60D1vKwRQsWTr5zuXUK?A-'
    'T#zeRGE!^v(K#ihGtv;is}Zi=ulJkmu8oS53xV10|o%V1!*F}-z)6xfx%;#cuLZF~EfK;K8U!k3Vur~|X8Beft{AvEq$pSI-$yT-'
    '6W85Oq$01|b)-N>u!eWw*6P`dn;0)HK#9~LlV`g9mUStT@Nnueb!WCMZ=$N_L6=O6<-T5)2-'
    'B#!9YBI?=&#Ex&rg?`(R^_egn(@Wu9w?asHxCHwaRh#&5zEibrcm56`%G*5wkZQGwm@-'
    '0Ai#r;Xy(qmU43Z=h_*)~7I0U}2HDEtNn5T$)-dw)|H2cbfE#HnR%xP7n%W_BW#9d-2wO?F-'
    'Z%J@*2VRMVp`M1EGwfyf!<1ofWE4zLxD?2@Ih5gTl%<rSkF-eiXS&7xzVE~q{1vd0mUL3aluJwWqfrY=dcsR{5LImkM-'
    'H74igu&dszJg`QB<uyu%=rf6Wa@qSJ7Ku0gDdZS3kgw$Q+P@sEvuq^Fx81qfQb-?nF5Bt%4EKLTxE9FH7|<>vnbTWW!WKbYt-'
    '6r2vX=Vy6D05MF&B5$>XPJF8MknZNa;3*hRU4HtBLsT`%7Aq8TZmn&Ry{c`2}R5D$3eBxSgwFv`Y6bQXeRP?OoAT|L~!hFyJ;37N'
    '^s+Ouo$M-XY4DbLE!1ry$6mE5>-'
    'LCSejU<)Vpv438VK<8Nln6z_@IWPIswP4XR}AA4B)ic){rFp0xQ_@%Sg=GT7dS@<vrz$<g*p|5(DA6@O#4U|w=m@&s}4d>S3n&0i'
    'ppLHAru}7tNfcr;io)`U##ZugyR44E*hE{@CwdII(1cNNYIk1!e@^cm4e#6%@X*^vla*Uc&QpvwmsU;-Kg!~{6E3#h^z'
)
heldout_text = unpack(
    'c-n1NO;a035WVv&w#sMo1IQ@{4mre7DK0sVv;%Bd?JBz~h;oud5mN;&;>4CKK%Bx<4nCwTgcbq%`U|t-'
    '59M|D?CJxb1Uoa`J+I$;JujT^RZvsDUw2e3*$pG@I;Fbv?RSof9p!BPu(7qV_2cHcljt~cVx4^ByxQ2p%C@I{_a%MBiAYx?9qS-'
    'biMOwvSXHXNUR`<NtgG^_bFk}SUsaX$%UD-cr07Tz?&-'
    'kMzf^s%>?M>w2*QMykl9zkPEGCTwSzEnSDja(<JM>=)}4xuV=Tpv7dT0zyub@~R#*1+*3A2?k+n=Zy>@i4R?#}aibD%Grjs?CsJ3'
    'P%W}Lmjrm-1jhrIS^((E{E<9A|G4)b>U#oXe{h)!fp?7hhvrej9t-lR6(*bMNcmz_|mFz~%VuTcDduyTi$L3U^!vL-Th=mh<m3I0'
    'Dpq#?zUleko<T#lJ$t*kx0M&3SGaXG!2{>hQrW=MNotz60@@v=j%X@EGCcSP~&R$S$PYq>J1xy830vYqiM<fqK2gz7@fpKIQ)%Qx'
    'ce?|>6m58SYCH>_y1VvqNBva{(0t)bcz(?@)ZE>1LX?lo!{bCh$!mJk-~Y1MS`t$`Z(PSq&<NtY82J9_#HLGJ-'
    'yia>y$FYy6Ib#dK>6r5;P;b2`V4_{NH>o9mtxg;P+%_H%EuW4I~*bM@b;H5a*TDv7xmJ2;jzYzFl!jbO)!do-sb8TwAWOj8Wq}|w'
    'ioRfp#WioY5XOV$KAco`(uHQ$6RMluk7K-'
    '%rnXBw&ZK2K#p6rx%>WcDw?%NRpFo(HELqfjj)RnABM{HTd@+S7CUljsp`x}29TSUvdI^wHcAVQyG9plpWf+%TB9m`sjZ+ebB57c'
    'uy4B>}S!aI=-'
    '&|#M(0z?C8B;+WmQ%?L5bxxSOU0jn^pckkGw;*<MNujIi(l_cCAtahEx<A7`Pk1MtB6@S0OBN!C$h`{>f<kDNu1P7ayi*AEHGSiZ'
    'A949Zbohjz7V^inXkXW@j3E<n1u_;f6)_5-t>DC_uT?BUZBE%ds6AGyB-'
    '1WMB2YypS=$=K5!+0Mgio2(c7o1OibE#Hqqqqv%4+;tzH^z0ijr88N(ewhdwSU2&CXE3Y0<$xQu5`{d#eJAi&epRzD?FnP#LRjV9'
    'OYiVo3|sA)x~>&I!mSD~17tTb8U)LEcSOE{C-sS!}44u*m6m@PgfizKCBb-'
    '%IM^7sTM1^(^#&LcXKgCASEHM_8`AM`jlF9@a00J6~HV3J+qzp5;qKYZe+A6zcngloR?!hal`dTg)SjY$W`qbPQ6;_A|15YjNj>7'
    'W#X3aJV_LP%qIIx8`ft!a(fk$d;Uyn3mo)O++sY3SMFCphFk{mRVQJhQA){_+Gp_&#5{8?^*Hu$MkGTOXHd^2HhhnbA_}ATx6?e;'
    '}x+=dA7v7q~&SIIN{Y0NT)I7E`aTsB}ki{Ym#45RWTv<wU7$a2|OUA?ydQEvfpSXn+LA?59$R^puAJ}xuaB6y`)7tPoHNK!fO6K4'
    'XJ$0R#hC+tHX{}JGyvir9{RVr#R{P+*4xCgTyyV6;J3r0swc#SiGD~C?1XkiLy%P8{DH467nX0i{z&L{4F5!Kqj$R%z~*E{p7Ld1'
    '<DuopqWi3E*)qsjO|ZLkClRRr0@ESFJu?9ngQZ~?AQYHfqVx+PQ{S`d!N5FW_DiDiXLx;V;nJEQaiJ+azR+XJ!D<NSNY-'
    '#sLs!2KKt+Tz@(l?()?+e_m@dJA_B5f9-hx)_76LTQ42qc@~2-'
    '(Z{y9p?3n247?Hj*PjY9Ebj1r?9mUU%4%miH3Ub}@teu~e6IRAP9tW!AOYR#sN{DggfBFEj1O'
)
challenge_text = unpack(
    'c-'
    'n1L%}yIJ5WeRrEO9OmKu?j{18PsS>fK39vdU&1tv5h!B&Y%jS_!qv0T2?m_D~Ui!cPdBSFqC;>Nn%vO#(f{@y>kn&5vhhMU>ay$W'
    'GGGv8svkyD#rIK9x7t%O8bFVr7g_JF?Xdm01;^byGzmlob^k5v$EmRTB|JiHd444P{jC$hv|=R5heK$=lM3FuJ9LX-Hz<(p6dMgf'
    'i}mz=$w_L12=gD#NfVWUCU?JGui$ACzhfquMf-'
    'i4vQ!Zi#jrw8*Zewp6IwEfpm|B&(%nbIqRFk$o0)yR$=k?+)#hzf#wChi=zSAbDowG4xM%$l3|4#;%WZYR7iOvMW0T{?;8b`l8@X'
    'U5_~IZDtF+ePYbHeahH|vman*y!(n9bIqeFSO+#GUE!~X7$_>}z03^|!*F(**}^v>%()POeO$~3*e7?Gp@*!W_!$5G=3E9ZYQ(u@'
    '+#^Ex+-Il0Hg=Nbjtr=JoKKJ9p*Z=gKf46Sfa-'
    'yB^7;*c?ai|R>j-Z!$YMUB+QBmz@(i;Yay2N9Qruaymx9iby|F3Z?%{9fzO(L}?kt@mg9mrOuS(yyV4L9g$hb4QdAZwld-jIzZqF'
    'UKW4s^9Pmt$0vmE6tZjU9`6c3~RAa@@{>h>v9cf`0JRnQ%QjiO8ESl9Q_aYdgQ+h^Nnyq%BjC%+{6FXQdQ-'
    '^kk$FLe8b|1(t6M>OHeWmeQDs9MS$gc>vU7rGM_bw}P;QKi)PJAdD>O=_>%;)1o~EaH@U2)-Zq`+>Y6zi~-'
    'LYuifT#4OVv<{W1=l*W)Q(eznim5%9*bRrfM1Zs8ZzOCaXGZ;E+I$~Xoab4Y^zh<Pj8IqLvWHZTl!#E{PmsFCY&|!_yH6>Zd9)mi'
    '&SB5#T!=zE7aFjFxjvL;YVUx}di@J&(`E08)T4-'
    '=8`ZsO{sKTJBiqXNa&=Aw3{%7F;UDu#pYSChqJo0eN8!aD;tK3r8mYZ)mqI@)bS~AIoDfLTbK6DG?zu27qC6<><K0J5S)-'
    'P5oy#aQ9(JuG2NERaUMPoH>5bB@ZISK8OqC|ObXVagTs-eR6t2tQQovh2cmRmwc#qR8*nA4?6e7Q%n7Ub8!s!oXrD<m!b14zHWQU'
)

print('Bundled synthetic datasets loaded:', len(seed_text.splitlines()), 'seed lines,',len(heldout_text.splitlines()),'held-out messages/lines,',len(challenge_text.splitlines()),'challenge messages/lines')


Bundled synthetic datasets loaded: 5 seed lines, 50 held-out messages/lines, 42 challenge messages/lines


## 2. Train a character-pattern classifier

**Classifier:** TF–IDF of character 2–4-grams + logistic regression; baseline: Latin letters → English, special Kazakh Cyrillic letters → Kazakh, other Cyrillic → Russian. We remove **all exact evaluation token forms** from training so the same word form cannot appear on both sides. A held-out synthetic set still does **not** prove real-world robustness.

In [2]:
import numpy as np
from collections import Counter
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

LANGS = ['en','kk','ru']
KAZAKH_LETTERS = set('әғқңөұүһіӘҒҚҢӨҰҮҺІ')

def parse_seeds(src):
    xs, ys, seen = [], [], set()
    for line in src.splitlines():
        if not line or line.startswith('#'): continue
        lang, words = line.split('|',1)
        for w in words.split():
            if (lang,w.casefold()) in seen: continue
            seen.add((lang,w.casefold()))
            xs.append(w); ys.append(lang)
    return xs, ys

def parse_messages(src):
    messages=[]
    for line in src.splitlines():
        if not line or line.startswith('#'): continue
        messages.append([tuple(tag.split(':',1))[::-1] for tag in line.split()])
    return messages

def baseline(word):
    if any('a' <= c.lower() <= 'z' for c in word): return 'en'
    if any(c in KAZAKH_LETTERS for c in word): return 'kk'
    return 'ru'

def score(messages, title):
    pairs = [p for m in messages for p in m]
    xs = [w for w,_ in pairs]
    true = [l for _,l in pairs]
    predicted = model.predict(xs).tolist()
    rules = [baseline(w) for w in xs]
    print('\n'+title+' —',len(messages),'messages,',len(pairs),'tokens')
    print('Rules baseline accuracy:', round(accuracy_score(true,rules),4), '| macro-F1:', round(f1_score(true,rules,labels=LANGS,average='macro'),4))
    print('Trained ML accuracy:',round(accuracy_score(true,predicted),4), '| macro-F1:',round(f1_score(true,predicted,labels=LANGS,average='macro'),4))
    print(classification_report(true,predicted,labels=LANGS,zero_division=0))
    mistakes = [(word,correct,guess) for word,correct,guess in zip(xs,true,predicted) if correct != guess]
    print('Example errors (word, expected, predicted):', mistakes[:12] if mistakes else 'none in this small synthetic test')
    return {'baseline_macro_f1':float(f1_score(true,rules,labels=LANGS,average='macro')),'model_macro_f1':float(f1_score(true,predicted,labels=LANGS,average='macro')),'errors':mistakes}

candidate_x, candidate_y = parse_seeds(seed_text)
heldout_messages = parse_messages(heldout_text)
challenge_messages = parse_messages(challenge_text)
reserved = {word.casefold() for msg in heldout_messages+challenge_messages for word,lang in msg}
pairs = [(word,lang) for word,lang in zip(candidate_x,candidate_y) if word.casefold() not in reserved]
X, y = [w for w,l in pairs], [l for w,l in pairs]
assert len({w.casefold() for w in X} & reserved) == 0
model = make_pipeline(
    TfidfVectorizer(analyzer='char',ngram_range=(2,4),lowercase=True,min_df=1),
    LogisticRegression(max_iter=1200,C=5.0,random_state=23)
).fit(X,y)
print('Training tokens:',len(X),'class counts:',dict(Counter(y)),'exact training-vs-test word overlap: 0')


Training tokens: 676 class counts: {'kk': 207, 'ru': 227, 'en': 242} exact training-vs-test word overlap: 0


## 3. Compare on unseen **synthetic** test messages and harder examples
These scores are not independent field validation. Look at the mistakes, not just overall accuracy.

In [3]:
heldout = score(heldout_messages, 'Held-out synthetic messages')
challenge = score(challenge_messages, 'Deliberately harder synthetic messages')


Held-out synthetic messages — 48 messages, 186 tokens
Rules baseline accuracy: 0.8118 | macro-F1: 0.8277
Trained ML accuracy: 0.9731 | macro-F1: 0.9756
              precision    recall  f1-score   support

          en       1.00      1.00      1.00        48
          kk       0.99      0.95      0.97        77
          ru       0.94      0.98      0.96        61

    accuracy                           0.97       186
   macro avg       0.97      0.98      0.98       186
weighted avg       0.97      0.97      0.97       186

Example errors (word, expected, predicted): [('туралы', 'kk', 'ru'), ('Достар', 'kk', 'ru'), ('ма', 'kk', 'ru'), ('Жоспар', 'kk', 'ru'), ('Здесь', 'ru', 'kk')]

Deliberately harder synthetic messages — 40 messages, 143 tokens
Rules baseline accuracy: 0.7413 | macro-F1: 0.7131
Trained ML accuracy: 0.9301 | macro-F1: 0.9287
              precision    recall  f1-score   support

          en       0.96      1.00      0.98        47
          kk       0.91      0.94

## 4. Try an actual school message

Edit `message` below and rerun. Labels are **model predictions**, not verified translations or calibrated confidence probabilities.

In [4]:
message = "Құрметті оқушылар, tomorrow we have a science quiz, поэтому дайындалыңдар!"
tokens = re.findall(r"[^\W\d_]+(?:[-'][^\W\d_]+)*", message, flags=re.UNICODE)
labels = model.predict(tokens)
from IPython.display import HTML, display
label_names={'kk':'Kazakh','ru':'Russian','en':'English'}
colors={'kk':'#eee2ff','ru':'#dcf4ff','en':'#e3f7e9'}
html='<div style="font-family:Arial,sans-serif;max-width:800px;line-height:3">'
for word,lang in zip(tokens,labels):
    html+=f'<span title="{label_names[lang]}" style="background:{colors[lang]};color:#1f2937;padding:8px;border-radius:8px;margin:3px;white-space:nowrap">{word} <small>{lang}</small></span>'
html+='</div>'
display(HTML(html))
print(list(zip(tokens, labels)))


[('Құрметті', np.str_('kk')), ('оқушылар', np.str_('kk')), ('tomorrow', np.str_('en')), ('we', np.str_('en')), ('have', np.str_('en')), ('a', np.str_('en')), ('science', np.str_('en')), ('quiz', np.str_('en')), ('поэтому', np.str_('ru')), ('дайындалыңдар', np.str_('kk'))]


## 5. Key limitations, impact & next experiment

- **Who uses it:** a teacher preparing a school notice for parents who read primarily Kazakh, Russian or English. We only highlight words for human follow-up; we do not translate.
- **Why it may help:** sentence-level models often assign one language to an entire mixed sentence; QosTil identifies switches at word level.
- **Failure cases:** names, transliteration, loanwords, slang, typos and shared Cyrillic vocabulary can be ambiguous. The classifier treats each word independently and cannot resolve context.
- **What is original:** a Kazakhstan-specific target user, three-language word tagging, bilingual/multilingual examples, head-to-head baseline and transparent synthetic error reporting.
- **Future validation:** recruit consenting Kazakh/Russian/English speakers, gather and anonymize non-sensitive sample notices with permission, annotate independently, freeze a test set, and report per-language macro-F1 and hard-case errors.

**Credits:** The content and synthetic examples were prepared with **ChatGPT** assistance. Model was trained with **scikit-learn**, using **Python, NumPy**, and built-in notebook facilities. No Tatoeba or Common Voice data was used. All listed datasets were created for this educational project, not third-party corpora.

**Official rules:** https://highschoolhackathon.org/ . Deadline: **November 15, 2026 11:59 PM ET** (November 16, 2026 09:59 AM Kazakhstan UTC+5).